# Generate a Protein Binder with BoltzGen

(Note: This notebook is designed for Google Colab. If you are running it locally, make sure to install the required dependencies and change the paths accordingly.)

In [1]:
pip install -qqq boltzgen py3Dmol wget

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.1/357.1 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.1/31.1 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.2/183.2 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.5 MB/s eta 0:00:00
   ━

## Setup Google Drive connection and working directory

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
## Make a working directory
import os
exercise_path = '/content/drive/MyDrive/datasmiths_bootcamp/day_3/exercise_1_boltzgen'
os.makedirs(exercise_path, exist_ok=True)
os.chdir(exercise_path)

## Grab the design specification files

You will need the target protein structure (a `.cif` file) and a design specification (a `.yaml` file). We'll grab these from the repo for the simple example.

In [5]:
experiment_path = 'vanilla_protein'

os.makedirs(experiment_path, exist_ok=True)

import wget
wget.download('https://github.com/colbyford/jcsu_datasmiths_structbio/raw/refs/heads/main/day_3/exercise_1_boltzgen/vanilla_protein/1g13.cif', out = experiment_path)
wget.download('https://github.com/colbyford/jcsu_datasmiths_structbio/raw/refs/heads/main/day_3/exercise_1_boltzgen/vanilla_protein/1g13prot.yaml', out = experiment_path)

os.listdir(experiment_path)
## Should return: ['1g13.cif', '1g13prot.yaml']

['1g13.cif', '1g13prot.yaml', '1g13 (1).cif', '1g13prot (1).yaml']

## Run the design process

`boltzgen run` takes the design specification .yaml and produces a set of ranked designs.

⚠️ it downloads model weights and data (~6GB) to `~/.cache` (or the directory set by `--cache` or `$HF_HOME`).

⚠️ If your run is ever interrupted, you can restart it with `--reuse`. No progress is lost.

Notes:
- `--num_designs` is the number of intermediate designs. In practice you will want between 10,000 - 60,000
- `--budget` is how many designs should be in the final diversity optimized set

In [7]:
%shell cd /content/drive/MyDrive/datasmiths_bootcamp/day_3/exercise_1_boltzgen/vanilla_protein/ && \
boltzgen run 1g13prot.yaml \
  --output workbench/test_run \
  --protocol protein-anything \
  --num_designs 5 \
  --budget 2


=== Configuring pipeline ===
Using dataset artifact: /root/.cache/huggingface/hub/datasets--boltzgen--inference-data/snapshots/c3d36fd276e9caf098c75d4113c6d5eb320b1a4c/mols.zip
Creating output directory: workbench/test_run
************** Checking design spec: 1g13prot.yaml **************
Total designed residues: 93
Design specification visualization is written to workbench/test_run/1g13prot.cif
*****************************************************************
Using kernels: False [device capability: (7, 5)]
Config overrides for protocol protein-anything: {}
Using 1 devices
Raw designs will be saved to: workbench/test_run/intermediate_designs
Using diffusion batch size: 1
Number of diffusion batches: 5
boltzgen1_diverse.ckpt: 100% 1.93G/1.93G [00:17<00:00, 110MB/s]
Using model artifact: /root/.cache/huggingface/hub/models--boltzgen--boltzgen-1/snapshots/c1be29e1f82ffcc72264f64b993c43fb4e0d17f0/boltzgen1_diverse.ckpt
boltzgen1_adherence.ckpt: 100% 1.93G/1.93G [00:18<00:00, 103MB/s]
Usin

## Evaluate the Design(s) and other outputs

In [9]:
import py3Dmol
from pathlib import Path

final_path = '/content/drive/MyDrive/datasmiths_bootcamp/day_3/exercise_1_boltzgen/vanilla_protein//workbench/test_run/final_ranked_designs/final_2_designs/'

## Collect CIF files
cif_files = sorted(Path(final_path).glob('*.cif'))

## Grid size
rows = 1
cols = 2

## Create grid viewer
view = py3Dmol.view(
    viewergrid=(rows, cols),
    width=1200,
    height=800
)

## Add each structure to a grid cell
for i, cif_path in enumerate(cif_files):
    r = i // cols
    c = i % cols

    if r >= rows:
        break

    with open(cif_path, "r") as f:
        cif_data = f.read()

    view.addModel(cif_data, "cif", viewer=(r, c))

    ## Show binder in yellow
    view.setStyle(
        {'chain':'A'},{'cartoon': {'color': 'yellow'}},
        viewer=(r, c)
    )

    ## Show target in cyan
    view.setStyle(
        {'chain':'B'},{'cartoon': {'color': 'cyan'}},
        viewer=(r, c)
    )

    view.zoomTo(viewer=(r, c))

## Show the grid
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Try Again!

Now that you've seen how BoltzGen works, find another target of interest and try to design some binders against it. You can use these [examples](https://github.com/HannesStark/boltzgen/tree/main/example) from the Boltz team for inspiration.